In [4]:
from model.train_models import train_evaluate_model
from utils.data_prep import get_clean_combined_data

In [5]:
xgb_params = {
    "max_depth": [3, 5, 7],
    "min_child_weight": [1, 3, 5],
    "max_delta_step": [0, 1, 5],
    "gamma": [0, 1, 3, 5],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_alpha": [0, 0.1, 1, 2],
    "reg_lambda": [1, 5, 10],
    "colsample_bylevel": [0.6, 0.8, 1.0],
}

In [6]:
winning_config = {
    "include_food": True,
    "include_rain": False,
    "include_text": True,
    "conflict_only": True,
    "k": 1.0,
    "event_col": "sub_event_type",
    "n_splits": 4,
    "use_pca": True,
}

# Locked, tuned hyperparameters from the winning run
# (acled_sub_food_text_conflict_pca_1_4, onset_aupr=0.4214) -
# no search, deterministic every time this cell runs.
winning_xgb_params = {
    "max_depth": 3,
    "min_child_weight": 5,
    "max_delta_step": 0,
    "gamma": 0,
    "learning_rate": 0.01,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 2.0,
    "reg_lambda": 5,
    "colsample_bylevel": 0.6,
}

data_sources = [
    src
    for src, include in zip(
        ["food", "rain", "text"],
        [
            winning_config["include_food"],
            winning_config["include_rain"],
            winning_config["include_text"],
        ],
    )
    if include
]

model_data, predictor_cols = get_clean_combined_data(
    data_sources=data_sources,
    k=winning_config["k"],
    event_col=winning_config["event_col"],
    conflict_only_embeddings=winning_config["conflict_only"],
)

final_params = {
    **winning_xgb_params,
    "k": winning_config["k"],
    "event_col": winning_config["event_col"],
    "n_splits": winning_config["n_splits"],
    "use_pca": winning_config["use_pca"],
}

results, best_params, shap_importance = train_evaluate_model(
    model_data,
    predictor_cols,
    final_params,
    best_params=True,  # skip RandomizedSearchCV entirely - use fixed params
    use_pca=winning_config["use_pca"],
    compute_shap=True,
    shap_sample_size=2000,
)

print(results)
print(shap_importance)

INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acled_sudan.csv.
INFO:Events processing:Data grouped by sub_event_type
INFO:Events processing:Escalation target set at 1.0 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:HDXClient:Reading local file: data/hdx/hdx_sudan_food_prices.csv
INFO:Data preparation:Food prices data processed.
INFO:Text processing:Reading local file: data/acled/acled_monthly_regional_embeddings_conflict_only.pkl
INFO:Data preparation:Notes data processed.
INFO:Text processing:PCA fit on train embeddings only: 768 raw dims -> 24 components for 90% variance.
INFO:Cross validation:Cross-validation testing splits:


--- Fold 1 ---
Train window: 2018-01 to 2018-12 (216 rows)
Test window:  2019-01 to 2019-12 (216 rows)
------------------------------
--- Fold 2 ---
Train window: 2018-01 to 2019-12 (432 rows)
Test window:  2020-01 to 2020-12 (216 rows)
------------------------------
--- Fold 3 ---
Train window: 2018-01 to 2020-12 (648 rows)
Test window:  2021-01 to 2021-12 (216 rows)
------------------------------
--- Fold 4 ---
Train window: 2018-01 to 2021-12 (864 rows)
Test window:  2022-01 to 2022-12 (216 rows)
------------------------------
{'optimal_threshold': '0.4407', 'n_predictors': 55, 'onset_aupr': '0.4214', 'onset_precision_class1': '0.3443', 'onset_recall_class1': '0.6364', 'onset_f1_class1': '0.4468', 'active_aupr': '0.3464', 'active_precision_class1': '0.3193', 'active_recall_class1': '0.6552', 'active_f1_class1': '0.4294'}
                               feature  mean_abs_shap
0                       rolling_std_6m       0.098518
1                      rolling_mean_6m       0.075928
2 